# finalAnswer_run — sinh submission THẬT cho public test

Notebook này **khác** `run_pipeline_kaggle.ipynb`:

| | `run_pipeline_kaggle` | `finalAnswer_run` (file này) |
|---|---|---|
| Đầu vào | `dev_150_locked.json` (cắt từ train) | câu hỏi public test của BTC |
| Mục đích | **đo** recall@5 nội bộ | **nộp bài** |
| Có gold? | có → tính được điểm | không → không tính được điểm |

### Vì sao có file này

Lần nộp 09/08 bị BTC từ chối chấm. Nguyên nhân: `submission.zip` chứa dự đoán cho
**150 câu dev nội bộ** thay vì câu hỏi public test. Định dạng file hoàn toàn đúng
(5 id/câu, không trùng, UTF-8, zip đúng cấu trúc) — chỉ sai bộ `question_id`.

`run_pipeline_kaggle` ở Bước 4 có gọi `build_submission(..., expected_qids=set(dev_data.keys()))`.
Guard **đã chạy và đã pass** — vì kỳ vọng truyền vào cũng chính là tập dev. Guard không
thể tự biết bạn đang nộp nhầm đề.

→ Bước 2 dưới đây là chốt chặn được thêm riêng để chặn đúng lỗi đó, không thể bỏ qua.

Bật **Internet: On** + **Accelerator: GPU T4 x2** trong Settings trước khi chạy.


## Cài package + trỏ vào dataset


In [ ]:
!pip install -q sentence-transformers

import sys
INPUT_DIR = "/kaggle/input/project-ir"   # đổi đúng slug thật -- xem !ls /kaggle/input
sys.path.append(INPUT_DIR)
!ls /kaggle/input


In [ ]:
import json, os
from pathlib import Path

from submission import build_submission, save_submission_zip, validate_submission_file
from rerank import load_reranker
from rerank_from_d import score_all_from_d, blend_bm25_first
import deep_chunk as DC


## Config

Hai đường dẫn đầu là thứ **phải sửa** trước khi chạy. Chúng phải trỏ tới dữ liệu
**public test**, không phải dev.


In [ ]:
# --- Dữ liệu public test ---
TEST_QUESTIONS_PATH = f"{INPUT_DIR}/public-official.json"     # 1000 câu đề thi, answer=null

# Candidate của D (335MB, giao 12/08). recall@100 = 0.9875
TEST_CANDIDATES_PATH = f"{INPUT_DIR}/bm25_top100_public.json"
CTX_DIR = f"{INPUT_DIR}/selected-contexts"     # tầng 2 cần toàn văn

# --- Dev: CHỈ để guard ở Bước 2. dev1000 bao trùm dev300 + dev150 nên chặn được hết ---
DEV_GOLD_PATH = f"{INPUT_DIR}/dev_1000_locked.json"

RERANKER_MODEL = "AITeamVN/Vietnamese_Reranker"
DEVICE = "cuda"
K = 5
N_BM25 = 2        # đỉnh ở cả dev300 lẫn public LB; quét lại sau deepchunk vẫn là 2

# --- E5 deepchunk: dev300 0.8883 -> 0.9083 (+2,00, bootstrap 96,6%) với đúng bộ này ---
# M=10 cho 0.9017 và rẻ hơn 2x; M=20 hơn +0,66, thắng M=10 ở cả 4 giá trị n.
# Lượt 15/08 chết vì Kaggle cắt ở 12h. Tách tầng 2 làm hai lượt bằng `skip`:
#   lượt A: SKIP, M_DOC = 0, 10   + SCORES_TANG1 = scores_public.json
#   lượt B: SKIP, M_DOC = 10, 20  + SCORES_TANG1 = scores_public_deep_M10_K20.json
# Có SCORES_TANG1 trên dataset thì tầng 1 KHÔNG chạy lại (~1h45m), đọc file là xong.
SKIP, M_DOC, K_CHUNK = 10, 20, 20
SCORES_TANG1 = f"{INPUT_DIR}/scores_public_deep_M10_K20.json"
VARIANT = "max"   # max(ce, ce_deep). Thắng "replace" ở cả 4 giá trị n
OUTPUT_DIR = "/kaggle/working/outputs"

# Đề thi có 1000 câu. Dưới ngưỡng này gần như chắc chắn đang trỏ nhầm vào file dev.
MIN_EXPECTED_QUESTIONS = 900

# Tầng 2 chỉ chạm CTX_DIR sau ~3h tầng 1 — kiểm ngay bây giờ, đừng chết ở giờ thứ 3.
assert os.path.isdir(CTX_DIR), f"KHÔNG thấy {CTX_DIR} — tầng 2 cần toàn văn corpus"


## Bước 1 — Đọc câu hỏi public test (tự dò schema)

`public-official.json` chưa được xác minh cấu trúc, nên hàm dưới nhận nhiều dạng
schema phổ biến và **báo lỗi rõ ràng** nếu không khớp dạng nào — thay vì âm thầm
trả về dict rỗng rồi sinh ra submission thiếu câu.


In [ ]:
QID_KEYS = ("question_id", "qid", "id", "q_id", "quest_id")
Q_KEYS   = ("question", "query", "text", "content", "cau_hoi", "question_text")


def load_questions_auto(path: str) -> dict:
    """
    Trả về {question_id(str): question_text(str)}.

    Nhận 3 dạng schema:
      A. {"qid": {"question": "..."}}          <- giống dev_150_locked
      B. {"qid": "câu hỏi ..."}                <- dict phẳng
      C. [{"question_id": "...", "question": "..."}, ...]   <- list bản ghi
    """
    with open(path, encoding="utf-8-sig") as f:   # utf-8-sig: nuốt luôn BOM nếu có
        raw = json.load(f)

    questions: dict[str, str] = {}

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                schema = "B"
                questions[str(k)] = v
            elif isinstance(v, dict):
                schema = "A"
                found = next((v[c] for c in Q_KEYS if isinstance(v.get(c), str)), None)
                if found is None:
                    raise ValueError(
                        f"Câu {k!r}: không tìm thấy trường câu hỏi. Các key có sẵn: "
                        f"{list(v)}. Bổ sung tên trường đúng vào Q_KEYS rồi chạy lại."
                    )
                questions[str(k)] = found
            else:
                raise ValueError(
                    f"Câu {k!r}: value kiểu {type(v).__name__}, không phải str/dict. "
                    f"Schema lạ -- kiểm tra lại file."
                )

    elif isinstance(raw, list):
        schema = "C"
        for i, item in enumerate(raw):
            if not isinstance(item, dict):
                raise ValueError(f"Phần tử {i} kiểu {type(item).__name__}, không phải dict.")
            qid = next((item[c] for c in QID_KEYS if c in item), None)
            q   = next((item[c] for c in Q_KEYS if isinstance(item.get(c), str)), None)
            if qid is None:
                raise ValueError(f"Phần tử {i}: không có trường id. Key có sẵn: {list(item)}")
            if q is None:
                raise ValueError(f"Phần tử {i}: không có trường câu hỏi. Key có sẵn: {list(item)}")
            if str(qid) in questions:
                raise ValueError(f"question_id {qid!r} xuất hiện nhiều hơn 1 lần -- BTC cấm trùng.")
            questions[str(qid)] = q
    else:
        raise ValueError(f"File gốc kiểu {type(raw).__name__}, phải là dict hoặc list.")

    if not questions:
        raise ValueError("Đọc được 0 câu hỏi -- file rỗng hoặc sai đường dẫn.")

    print(f"Nhận diện schema: {schema}")
    print(f"Đọc được {len(questions)} câu hỏi")
    empty = [q for q, t in questions.items() if not t.strip()]
    if empty:
        raise ValueError(f"{len(empty)} câu có nội dung rỗng: {empty[:5]}")
    return questions


test_questions = load_questions_auto(TEST_QUESTIONS_PATH)

_k0 = next(iter(test_questions))
print(f"\nVí dụ -- {_k0}: {test_questions[_k0][:100]}")


## Bước 2 — CHỐT CHẶN: đây có đúng là đề thi không?

Ô này tồn tại vì đúng một lý do: lần nộp 09/08 đã nộp nhầm tập dev. **Không được
bỏ qua ô này.** Nếu nó raise, đừng sửa nó — sửa đường dẫn ở Config.


In [ ]:
def assert_khong_phai_dev(test_qids: set, dev_gold_path: str, min_expected: int):
    loi = []

    # 1. Trùng khít tập dev -> đúng y hệt lỗi lần trước
    try:
        with open(dev_gold_path, encoding="utf-8-sig") as f:
            dev_qids = set(json.load(f))
    except FileNotFoundError:
        print(f"[bỏ qua đối chiếu dev] không thấy {dev_gold_path}")
        dev_qids = set()

    if dev_qids:
        chung = test_qids & dev_qids
        if test_qids == dev_qids:
            loi.append(
                f"question_id TRÙNG KHÍT 100% với dev_150_locked ({len(chung)} câu). "
                f"Đây CHÍNH XÁC là lỗi đã làm bài nộp 09/08 bị từ chối. "
                f"TEST_QUESTIONS_PATH đang trỏ vào dev, không phải đề thi."
            )
        elif chung:
            loi.append(
                f"{len(chung)}/{len(test_qids)} câu trùng với dev_150_locked. Dev được cắt từ "
                f"train nên KHÔNG được xuất hiện trong đề thi -- kiểm tra lại nguồn dữ liệu."
            )

    # 2. Số câu quá ít -> gần như chắc chắn đang cầm nhầm file
    if len(test_qids) < min_expected:
        loi.append(
            f"Chỉ có {len(test_qids)} câu, ít hơn ngưỡng {min_expected}. Đề public test dự kiến "
            f"lớn hơn nhiều. Nếu đề thật đúng là nhỏ như vậy, hạ MIN_EXPECTED_QUESTIONS -- "
            f"nhưng hãy xác minh với BTC trước."
        )

    if loi:
        raise RuntimeError(
            "DỪNG LẠI -- dữ liệu đầu vào không giống đề thi:\n"
            + "\n".join(f"  [{i+1}] {e}" for i, e in enumerate(loi))
            + "\n\nSửa TEST_QUESTIONS_PATH / TEST_CANDIDATES_PATH ở Config. Đừng sửa ô guard này."
        )

    print(f"OK -- {len(test_qids)} câu, không trùng dev. Đi tiếp được.")


assert_khong_phai_dev(set(test_questions), DEV_GOLD_PATH, MIN_EXPECTED_QUESTIONS)


## Bước 3 — Đọc candidate của D + kiểm tra phủ

Câu nào không có candidate thì sẽ không có đáp án → thiếu câu trong submission.
Bắt ngay tại đây thay vì để lộ ra lúc nộp.


In [ ]:
with open(TEST_CANDIDATES_PATH, encoding="utf-8-sig") as f:
    test_candidates = json.load(f)   # {"qid": [{"doc_id":.., "top_chunks":[{"text":..}]}]}

test_candidates = {str(k): v for k, v in test_candidates.items()}
print(f"Candidate: {len(test_candidates)} câu")

thieu = set(test_questions) - set(test_candidates)
if thieu:
    raise ValueError(
        f"{len(thieu)} câu KHÔNG có candidate -- sẽ thiếu câu trong submission. "
        f"Đẩy về D bổ sung: {sorted(thieu)[:10]}"
    )

thua = set(test_candidates) - set(test_questions)
if thua:
    print(f"[cảnh báo] {len(thua)} câu có candidate nhưng không có trong đề -- sẽ bỏ qua, không nộp.")

rong = [q for q in test_questions if not test_candidates.get(q)]
if rong:
    raise ValueError(f"{len(rong)} câu có candidate RỖNG: {rong[:10]}")

so_cand = [len(test_candidates[q]) for q in test_questions]
total_pairs = sum(
    len(c.get("top_chunks", [])) for q in test_questions for c in test_candidates[q]
)
print(f"Candidate/câu: min {min(so_cand)}, max {max(so_cand)}, tb {sum(so_cand)/len(so_cand):.1f}")
print(f"Tổng cặp cross-encoder phải chấm: {total_pairs:,} (~{total_pairs/len(test_questions):.0f}/câu)")


## Bước 4 — Load reranker (kèm đếm tham số thật, ngân sách ≤ 3.0B)


In [ ]:
score_fn = load_reranker(RERANKER_MODEL, device=DEVICE)


## Bước 5 — Rerank toàn bộ đề thi

Không có gold nên **không in ra recall được**. Con số tin cậy duy nhất về chất
lượng model là recall@5 = 0.8567 đo trên dev bằng `run_pipeline_kaggle`.

Chạy lâu hơn dev nhiều lần — số cặp tỉ lệ thuận với số câu.


In [ ]:
import time

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# --- TẦNG 1: candidate của D, y như cũ ---
if os.path.exists(SCORES_TANG1):
    scores = json.load(open(SCORES_TANG1, encoding="utf-8"))
    print(f"đọc lại điểm cũ: {SCORES_TANG1} ({len(scores)} câu) -- KHÔNG chạy lại tầng 1")
    assert set(scores) == set(test_questions), "file điểm cũ không khớp bộ câu hỏi"
else:
    t0 = time.time()
    scores = score_all_from_d(test_questions, test_candidates, score_fn)
    print(f"tầng 1 xong {len(scores)} câu trong {(time.time()-t0)/60:.0f} phút")

    # LƯU NGAY. Quy tắc 2: tầng 2 hỏng thì lượt GPU này vẫn phải còn lại thứ gì đó.
    with open(f"{OUTPUT_DIR}/scores_public.json", "w", encoding="utf-8") as f:
        json.dump(scores, f, ensure_ascii=False)
    print("ĐÃ LƯU scores_public.json -- TẢI VỀ dù phần sau có hỏng")

# Hết dùng text candidate rồi, phần sau chỉ cần doc_id. Bỏ đi bớt ~2GB RAM cho tầng 2.
bm25_rank = {q: [str(c["doc_id"]) for c in test_candidates[q]] for q in test_questions}
for _v in test_candidates.values():
    for _c in _v:
        _c.pop("top_chunks", None)

# --- TẦNG 2: chấm sâu M_DOC văn bản đầu, K_CHUNK đoạn mỗi văn bản ---
# try/except vì commit LỖI thì Kaggle không xuất bản output -- mất luôn scores_public.json
# vừa lưu ở trên. Tầng 2 chết thì `scores` giữ nguyên giá trị tầng 1, và VARIANTS["max"]
# tự rơi về "ce" khi không có ce_deep -> vẫn ra đúng submission baseline, không mất lượt GPU.
try:
    n2 = DC.count_deep_chunks(test_questions, scores, CTX_DIR, M_DOC, K_CHUNK, skip=SKIP)
    print(f"tầng 2 sẽ chấm {n2:,} đoạn ({n2/len(test_questions):.0f}/câu)")
    assert n2 < 600_000, "quá nhiều -- kiểm lại M_DOC/K_CHUNK trước khi đốt GPU"

    t0 = time.time()
    scores = DC.deepen_all(test_questions, scores, CTX_DIR, score_fn, M_DOC, K_CHUNK, every=25, skip=SKIP)
    print(f"tầng 2 xong trong {(time.time()-t0)/60:.0f} phút")

    with open(f"{OUTPUT_DIR}/scores_public_deep_M{M_DOC}_K{K_CHUNK}.json", "w", encoding="utf-8") as f:
        json.dump(scores, f, ensure_ascii=False)
except Exception as e:
    print(f"[TẦNG 2 HỎNG] {type(e).__name__}: {e}")
    print("-> chạy tiếp bằng điểm tầng 1. Submission ra sẽ là baseline 0.8573, KHÔNG phải 0.9083.")

# --- Chốt top-5 ---
predicted_test = {
    q: blend_bm25_first(DC.rank_by(scores[q], VARIANT), bm25_rank[q], k=K, n_bm25=N_BM25)
    for q in test_questions
}

thieu = [q for q, v in predicted_test.items() if not v]
if thieu:
    print(f"[cảnh báo] {len(thieu)} câu rỗng -- guard sẽ bù từ fallback.")


## Bước 6 — Build submission + validate + đóng gói

`ranked_fallback_dict` là thứ tự gốc của D, dùng để bù khi rerank trả về dưới 5
document phân biệt. `expected_qids` lần này là **đề thi thật**, không phải dev.


In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

fallback = {
    q: [str(c["doc_id"]) for c in test_candidates[q]]   # cả 100, không cắt 5 -- bù mới đủ
    for q in test_questions
}

expected = set(test_questions)

sub = build_submission(
    predicted_test,
    ranked_fallback_dict=fallback,
    k=K,
    expected_qids=expected,
)

zip_path = save_submission_zip(sub, out_dir=OUTPUT_DIR)

errors = validate_submission_file(
    os.path.join(OUTPUT_DIR, "submission.json"),
    expected_qids=expected,
    k=K,
)

if errors:
    print("CÓ LỖI -- KHÔNG ĐƯỢC NỘP:")
    for e in errors:
        print(" -", e)
else:
    print(f"OK -- đã lưu {zip_path}")


## Bước 7 — Kiểm tra lần cuối trên chính file zip

Mở lại đúng file sắp nộp và soi từ trong ra, không tin biến trong RAM.


In [ ]:
import zipfile

with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    assert names == ["submission.json"], f"Zip phải chứa DUY NHẤT submission.json, đang có: {names}"
    raw = zf.read("submission.json")

assert raw[:3] != b"\xef\xbb\xbf", "File có BOM -- BTC yêu cầu UTF-8 không BOM"
data = json.loads(raw.decode("utf-8"))

assert set(data) == expected, (
    f"Bộ question_id trong zip KHÔNG khớp đề thi "
    f"(thiếu {len(expected - set(data))}, thừa {len(set(data) - expected)})"
)
for q, v in data.items():
    a = v["answer"]
    assert isinstance(q, str) and isinstance(a, list), f"{q}: sai kiểu dữ liệu"
    assert 1 <= len(a) <= K,        f"{q}: có {len(a)} id"
    assert len(a) == len(set(a)),   f"{q}: có id trùng"
    assert all(isinstance(x, str) for x in a), f"{q}: có id không phải string"

du5 = sum(1 for v in data.values() if len(v["answer"]) == K)
print(f"Số câu           : {len(data)}")
print(f"Đủ {K} id         : {du5}/{len(data)}")
print(f"Kích thước zip   : {os.path.getsize(zip_path):,} bytes")
print(f"\nTẤT CẢ KIỂM TRA ĐỀU PASS -- tải {zip_path} về máy rồi nộp.")
print("Kaggle không giữ /kaggle/working qua các session, tải ngay.")
